# 02 — Minimal viable simulation (independent, Gaussian)

**Phase 2 companion notebook.**

Goals:
1. Generate m independent Gaussian t-stats (pi1 non-null, effect size delta).
2. Apply Bonferroni, Holm, BH, BY (from `src/corrections.py`).
3. Monte Carlo: empirical FWER, FDR, power as a function of m.
4. Key chart: uncorrected FWER blows past alpha; Bonferroni/Holm stay at or below it.

**Definition of done:** uncorrected FWER ≈ 1 - (1 - alpha)^m. If it isn't, stop and fix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys
sys.path.insert(0, '..')

from src.simulate import run_simulation
from src.corrections import bonferroni, holm, benjamini_hochberg, benjamini_yekutieli, implied_threshold

rng = np.random.default_rng(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# ── Parameter grid ───────────────────────────────────────────────────────────
M_VALUES = [10, 30, 50, 100, 200, 300, 500, 1000]
ALPHA    = 0.05
N_REPS   = 2000   # ±1 pp std-err on a 5% rate

corrections = {
    "uncorrected": lambda p, a: np.asarray(p) <= a,
    "bonferroni":  bonferroni,
    "holm":        holm,
    "bh":          benjamini_hochberg,
    "by":          benjamini_yekutieli,
}

# ── All-null sweep (pi1=0, delta=0): only FWER meaningful ────────────────────
results_null = {}
for m in M_VALUES:
    results_null[m] = run_simulation(
        m=m, pi1=0.0, delta=0.0, n_reps=N_REPS,
        corrections=corrections, alpha=ALPHA, rng=rng
    )

# Sanity check: empirical uncorrected FWER should ≈ 1-(1-α)^m
print(f"{'m':>6}  {'uncorr':>8}  {'theory':>8}  {'bonf':>8}  {'holm':>8}  {'bh':>8}  {'by':>8}")
for m in M_VALUES:
    r = results_null[m]
    theory = 1 - (1 - ALPHA) ** m
    print(f"{m:6d}  {r['uncorrected']['fwer']:8.3f}  {theory:8.3f}  "
          f"{r['bonferroni']['fwer']:8.3f}  {r['holm']['fwer']:8.3f}  "
          f"{r['bh']['fwer']:8.3f}  {r['by']['fwer']:8.3f}")

In [ ]:
# ── Snapshot table: m=300, all corrections ───────────────────────────────────
M_SNAP = 300
rows = []
for name in corrections:
    r_null  = results_null[M_SNAP][name]
    r_mixed = results_mixed[M_SNAP][name]
    rows.append({
        "Method":          name,
        "FWER (all-null)": round(r_null["fwer"],   3),
        "FWER (mixed)":    round(r_mixed["fwer"],  3),
        "FDR  (mixed)":    round(r_mixed["fdr"],   3),
        "Power (mixed)":   round(r_mixed["power"], 3),
    })

df_snap = pd.DataFrame(rows).set_index("Method")
print(f"m={M_SNAP},  π₁={PI1},  δ={DELTA},  α={ALPHA},  n_reps={N_REPS}\n")
print(df_snap.to_string())

In [ ]:
# ── Mixed-signal sweep (pi1=0.1, delta=3): FWER, FDR, Power vs m ────────────
PI1   = 0.1
DELTA = 3.0

results_mixed = {}
for m in M_VALUES:
    results_mixed[m] = run_simulation(
        m=m, pi1=PI1, delta=DELTA, n_reps=N_REPS,
        corrections=corrections, alpha=ALPHA, rng=rng
    )

colors = {
    "uncorrected": "#e74c3c",
    "bonferroni":  "#2980b9",
    "holm":        "#27ae60",
    "bh":          "#8e44ad",
    "by":          "#f39c12",
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (metric, ylabel) in zip(axes, [("fwer", "FWER"), ("fdr", "FDR"), ("power", "Power (TPR)")]):
    for name, color in colors.items():
        vals = [results_mixed[m][name][metric] for m in M_VALUES]
        ax.plot(M_VALUES, vals, "o-", color=color, lw=2, label=name.capitalize())
    if metric in ("fwer", "fdr"):
        ax.axhline(ALPHA, color="black", lw=1, linestyle=":", label=f"α={ALPHA}")
    ax.set_xscale("log")
    ax.set_xlabel("$m$  (log scale)")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())

axes[0].legend(fontsize=8)
fig.suptitle(f"Mixed signal: π₁ = {PI1},  δ = {DELTA}  —  all methods vs m", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Table: implied Bonferroni t-stat threshold vs m ─────────────────────────
df_thresh = pd.DataFrame({
    "m":                M_VALUES,
    "t* (Bonferroni)":  [round(implied_threshold(m, ALPHA, "bonferroni"), 3) for m in M_VALUES],
    "√(2 ln m)":        [round(np.sqrt(2 * np.log(m)), 3) for m in M_VALUES],
}).set_index("m")
print(df_thresh.to_string())
print()
print("t*(Bonferroni) = Φ⁻¹(1 - α/2m)  vs  expected max of m standard-normals ≈ √(2 ln m)")

In [ ]:
# ── Figure 1: The money chart — FWER vs number of tests ─────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

fwer      = {name: [results_null[m][name]["fwer"] for m in M_VALUES] for name in corrections}
theory_fwer = [1 - (1 - ALPHA) ** m for m in M_VALUES]

ax.plot(M_VALUES, fwer["uncorrected"], "o-", color="#e74c3c", lw=2,
        label="Uncorrected (empirical)", zorder=4)
ax.plot(M_VALUES, theory_fwer, "--", color="#e74c3c", lw=1.5, alpha=0.6,
        label=r"Theoretical $1-(1-\alpha)^m$")
ax.plot(M_VALUES, fwer["bonferroni"], "s-", color="#2980b9", lw=2, label="Bonferroni")
ax.plot(M_VALUES, fwer["holm"],       "^-", color="#27ae60", lw=2, label="Holm")
ax.plot(M_VALUES, fwer["bh"],         "D-", color="#8e44ad", lw=2, label="Benjamini–Hochberg")
ax.plot(M_VALUES, fwer["by"],         "P-", color="#f39c12", lw=2, label="Benjamini–Yekutieli")
ax.axhline(ALPHA, color="black", lw=1, linestyle=":", label=f"α = {ALPHA}")

ax.set_xscale("log")
ax.set_xlabel("Number of tests $m$  (log scale)", fontsize=12)
ax.set_ylabel("FWER  (empirical)", fontsize=12)
ax.set_title(
    "Multiple-testing corrections keep FWER ≤ α;\n"
    "uncorrected FWER → 1 as $m$ grows  (all-null, π₁ = 0)",
    fontsize=13,
)
ax.set_ylim(-0.02, 1.05)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.legend(loc="center right", fontsize=10)
plt.tight_layout()
plt.show()